# 00 — Fetch the full 96-month raw panel

**Goal:** ensure every monthly RTT full extract from **2018-04 to 2026-03** exists on disk.
Driven by `nhs_manifest.json` (96 verified NHS England URLs + byte sizes, recovered from the
project's earlier A100 line). Safe to rerun — existing files are skipped, so it is resumable.

In [ ]:
import json
import os
import zipfile
import requests

RAW = "../data/nhs_rtt_referral_to_treatment"
ARCH = "../data/archives/rtt"
os.makedirs(RAW, exist_ok=True)
os.makedirs(ARCH, exist_ok=True)

manifest = json.load(open("nhs_manifest.json"))["files"]
missing = [f for f in manifest if not os.path.exists(f"{RAW}/rtt_{f['month']}.csv")]
print(f"manifest: {len(manifest)} months ({manifest[0]['month']} .. {manifest[-1]['month']})")
print(f"already on disk: {len(manifest) - len(missing)} | missing: {len(missing)}")
print("missing months:", [f["month"] for f in missing][:40])

In [ ]:
# download each missing month's zip (skipped if archived already), extract the CSV
for f in missing:
    zp = f"{ARCH}/rtt-{f['month']}.zip"
    if not os.path.exists(zp):
        r = requests.get(f["url"], timeout=180)
        r.raise_for_status()
        if len(r.content) != f["content_length"]:
            print(f"  note {f['month']}: size {len(r.content)} != manifest {f['content_length']} (NHS revision?)")
        open(zp, "wb").write(r.content)
    with zipfile.ZipFile(zp) as z:  # extract the (largest) CSV member under our canonical name
        member = max((n for n in z.namelist() if n.lower().endswith(".csv")),
                     key=lambda n: z.getinfo(n).file_size)
        with z.open(member) as src, open(f"{RAW}/rtt_{f['month']}.csv", "wb") as dst:
            dst.write(src.read())
    print(f"fetched {f['month']}", flush=True)
print("done")

In [ ]:
# final verification: unbroken monthly sequence, readable headers
import glob
csvs = sorted(glob.glob(f"{RAW}/rtt_*.csv"))
months = [os.path.basename(p)[4:11] for p in csvs]
print(f"on disk: {len(csvs)} files ({months[0]} .. {months[-1]})")
expected = {f["month"] for f in manifest}
assert expected.issubset(set(months)), f"still missing: {sorted(expected - set(months))}"
print("all 96 manifest months present - OK")